# 01 - Data Acquisition
Project: Finnish NHL Players: A Zone-Level Shooting Efficiency Analysis

## Purpose
Load raw shot data from MoneyPuck and player biography data,
validate both datasets, merge them, and save a clean combined
dataset to data/processed/ for use in subsequent notebooks.

## Inputs
- data/raw/shots_2022.csv
- data/raw/shots_2023.csv
- data/raw/shots_2024.csv
- data/raw/allPlayersLookup.csv

## Output
- data/processed/shots_combined.csv

In [1]:
import pandas as pd
import os
from pathlib import Path

RAW_DATA = Path('../data/raw')
PROCESSED_DATA = Path('../data/processed')
PROCESSED_DATA.mkdir(exist_ok=True)

print(f"Raw data path {RAW_DATA.resolve()}") # Testing whether the data paths work as expected
print(f"Processed data path {PROCESSED_DATA.resolve()}")



Raw data path C:\Users\juuso\analysis-on-finnish-nhlers-shooting-efficiency\data\raw
Processed data path C:\Users\juuso\analysis-on-finnish-nhlers-shooting-efficiency\data\processed


In [2]:
seasons = ['2022', '2023', '2024']
dfs = []

for season in seasons:
    df = pd.read_csv(RAW_DATA / f'shots_{season}.csv')
    df['season'] = season # renaming the "season" headers to be able to differentiate years
    dfs.append(df)
    print(f"Testing correctness with amount of rows for season {season} -> {len(df)} rows loaded") 

combined_shots = pd.concat(dfs, ignore_index = True)
print(f"Testing correctness with amount of shots for combined shots -> {len(combined_shots)} and columns -> {combined_shots.shape[1]}")

Testing correctness with amount of rows for season 2022 -> 122026 rows loaded
Testing correctness with amount of rows for season 2023 -> 122472 rows loaded
Testing correctness with amount of rows for season 2024 -> 119870 rows loaded
Testing correctness with amount of shots for combined shots -> 364368 and columns -> 137


In [3]:
player_bios = pd.read_csv(RAW_DATA / 'allPlayersLookup.csv')
print(f"Testing correctness with amount of rows for the data -> {len(player_bios)} rows loaded")

Testing correctness with amount of rows for the data -> 3400 rows loaded


In [4]:
combined_shots = combined_shots.merge(player_bios[['playerId', 'nationality']], left_on = 'shooterPlayerId', right_on = 'playerId', how = 'left')

print(f"Rows after merge: {len(combined_shots)}")
print(f"Columns after merge: {combined_shots.shape[1]}") # Should be +2 compared to value gotten before merge
print(f"Nationality null values: {combined_shots['nationality'].isnull().sum()}")
print(f"Top 10 nationalities by shot volume: {combined_shots['nationality'].value_counts().head(10)}")                                      

Rows after merge: 364368
Columns after merge: 139
Nationality null values: 3066
Top 10 nationalities by shot volume: nationality
CAN    148271
USA    106737
SWE     35236
RUS     21913
FIN     17396
CZE      9419
CHE      7527
DEU      4202
DNK      2425
SVK      2375
Name: count, dtype: int64


# 02 - Data Validation

Before saving, we validate that the combined dataset looks as expected.
Three checks are performed:
1. Playoff games are removed - regular season only
2. xG values are within expected range [0, 1]
3. Null nationality values are documented and removed
4. Columns with all null values are removed

In [5]:
before = len(combined_shots)
combined_shots = combined_shots[combined_shots['isPlayoffGame'] == 0]
after = len(combined_shots)

print(f"Playoff rows removed: {before - after}")
print(f"Rows remaining: {after}")

Playoff rows removed: 23379
Rows remaining: 340989


In [6]:
xg_min = combined_shots['xGoal'].min()
xg_max = combined_shots['xGoal'].max()
xg_nulls = combined_shots['xGoal'].isnull().sum()

print(f"xG minimum value: {xg_min:.4f}")
print(f"xG maximum value: {xg_max:.4f}")
print(f"xG null values: {xg_nulls}")

if xg_min >= 0 and xg_max <= 1 and xg_nulls == 0:
    print("xG values look valid")
else:
    print("xG values outside expected range")

xG minimum value: 0.0014
xG maximum value: 0.9767
xG null values: 0
xG values look valid


In [7]:
nationality_nulls = combined_shots['nationality'].isnull().sum()
print(f"Rows with missing nationality: {nationality_nulls}")
print(f"This represents {nationality_nulls / len(combined_shots) * 100:.2f}% of all rows")

combined_shots = combined_shots.dropna(subset=['nationality'])
print(f"Rows after removing null nationalities: {len(combined_shots)}")

Rows with missing nationality: 2848
This represents 0.84% of all rows
Rows after removing null nationalities: 338141


In [8]:
columns_before = combined_shots.shape[1]
combined_shots = combined_shots.dropna(axis=1, how='all')
columns_after = combined_shots.shape[1]

print(f"Columns removed: {columns_before - columns_after}")
print(f"Columns remaining: {columns_after}")

Columns removed: 13
Columns remaining: 126


# 03 - Saving the processed data

In [9]:
output_path = PROCESSED_DATA / 'shots_combined.csv'
combined_shots.to_csv(output_path, index=False)

print(f"Dataset saved to: {output_path.resolve()}")
print(f"Final dataset: {len(combined_shots)} rows, {combined_shots.shape[1]} columns")

Dataset saved to: C:\Users\juuso\analysis-on-finnish-nhlers-shooting-efficiency\data\processed\shots_combined.csv
Final dataset: 338141 rows, 126 columns


# 04 - Summary

This notebook loaded and validated the raw shot data from MoneyPuck.

Input:
- shots_2022.csv, shots_2023.csv, shots_2024.csv
- allPlayersLookup.csv (player biography data to get nationalities merged with the shot data)

Processing steps:
1. Merged nationality data from player biographies
2. Removed playoff games (23379 rows)
3. Removed rows with missing nationality (2848 rows)

Output:
- data/processed/shots_combined.csv (338141 rows, 139 columns)